# ObjectLab ArtAug image-to-image pilot

This notebook uses the released DiffSynth-Studio FLUX.1-Kontext ArtAug adapter to make a small set of **synthetic training-only** image edits. It is a pilot, not a reproduction of the original paper's training method.

**Current guard:** the user finalized four class-specific prompts, now recorded below. `PROMPTS_FINALIZED` is true, but generation remains off with `RUN_GENERATION = False` until the Colab GPU and gated Hugging Face access are confirmed. You can run source selection, hash verification, preview, and preflight without generating.

The base FLUX.1-Kontext-dev checkpoint is gated and non-commercial. Accept its terms before requesting access. Use a Colab GPU with CUDA/BF16; this notebook does not promise compatibility with low-memory runtimes.


In [ ]:
# Upload ObjectLab_Source.zip using Colab's Files panel or the chooser below.
from pathlib import Path
import subprocess, sys, os, json, csv, hashlib, random, time, shutil, urllib.request, zipfile
from google.colab import files
archive_path = Path("/content/ObjectLab_Source.zip")
if not archive_path.is_file():
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise RuntimeError("Upload exactly one ObjectLab_Source.zip archive")
    archive_path = Path(zip_names[0])
extract_root = Path("/content/objectlab_source_upload")
extract_root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive_path) as archive:
    base = extract_root.resolve()
    for member in archive.infolist():
        target = (extract_root / member.filename).resolve()
        if not target.is_relative_to(base) or (member.external_attr >> 16) & 0o170000 == 0o120000:
            raise RuntimeError(f"Unsafe archive entry: {member.filename}")
    archive.extractall(extract_root)
REPO_DIR = extract_root / "ObjectLab"
assert (REPO_DIR / "results/dataset_manifest.csv").is_file(), "Frozen manifest missing"
assert (REPO_DIR / "data/download_manifest.json").is_file(), "Download manifest missing"
print("Source archive ready at", REPO_DIR)
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024*1024), b""):
            digest.update(block)
    return digest.hexdigest()


In [ ]:
# Select a reproducible set of 5 TRAIN images per class from the frozen manifests.
CLASS_ORDER = ["bottles", "cups", "pens", "scissors"]
PILOT_PER_CLASS = 5
PILOT_SEED = 4127
manifest_path = REPO_DIR / "results/dataset_manifest.csv"
manifest_bytes = manifest_path.read_bytes()
SOURCE_MANIFEST_SHA256 = hashlib.sha256(manifest_bytes).hexdigest()
with manifest_path.open("r", newline="", encoding="utf-8-sig") as stream:
    frozen_rows = list(csv.DictReader(stream))
download_rows = json.loads((REPO_DIR / "data/download_manifest.json").read_text(encoding="utf-8"))
download_by_path = {row["path"].replace("\\", "/"): row for row in download_rows}
PILOT_SOURCE_QC_EXCLUSIONS = {
    "train/pens/77.jpg": "Manual source review: dominant scissors plus pen make the class visually ambiguous for the pen pilot. Excluded from this pilot only; frozen dataset and splits remain unchanged."
}
train_by_class = {label: [] for label in CLASS_ORDER}
for row in frozen_rows:
    if row["split"] != "train":
        continue
    label, path = row["class"], row["path"].replace("\\", "/")
    if label not in train_by_class:
        raise ValueError(f"Unexpected class in frozen manifest: {label}")
    drive = download_by_path.get(path)
    if drive is None or drive["sha256"].lower() != row["sha256"].lower():
        raise ValueError(f"Download manifest hash mismatch: {path}")
    if path not in PILOT_SOURCE_QC_EXCLUSIONS:
        train_by_class[label].append({**row, **drive, "path": path})
chooser = random.Random(PILOT_SEED)
SELECTED = []
for label in CLASS_ORDER:
    candidates = sorted(train_by_class[label], key=lambda item: item["path"])
    if len(candidates) < PILOT_PER_CLASS:
        raise RuntimeError(f"Not enough training photos for {label}")
    SELECTED.extend(chooser.sample(candidates, PILOT_PER_CLASS))
assert len(SELECTED) == 20 and all(row["split"] == "train" for row in SELECTED)
print("Frozen manifest SHA256:", SOURCE_MANIFEST_SHA256)
print("Pilot-only source QC exclusions:", PILOT_SOURCE_QC_EXCLUSIONS)
for label in CLASS_ORDER:
    print(label, sum(row["class"] == label for row in SELECTED))


In [ ]:
# Download only the chosen 20 public Drive files. Every byte is checked against the frozen SHA256.
from PIL import Image, ImageOps
SOURCE_DIR = Path("/content/artaug_pilot/source")
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
for row in SELECTED:
    target = SOURCE_DIR / row["path"]
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists() or sha256_file(target) != row["sha256"]:
        url = f"https://drive.usercontent.google.com/download?id={row['id']}&export=download&confirm=t"
        with urllib.request.urlopen(url, timeout=90) as response:
            payload = response.read()
        actual = hashlib.sha256(payload).hexdigest()
        if actual != row["sha256"]:
            raise RuntimeError(f"Drive bytes failed frozen hash check for {row['path']}: {actual}")
        target.write_bytes(payload)
    with Image.open(target) as im:
        im.verify()
    print("verified", row["class"], row["path"], row["sha256"][:12])


In [ ]:
# Preview source photos before deciding the final prompts.
import matplotlib.pyplot as plt
fig, axes = plt.subplots(4, PILOT_PER_CLASS, figsize=(15, 10))
for ax, row in zip(axes.flat, SELECTED):
    with Image.open(SOURCE_DIR / row["path"]) as im:
        ax.imshow(ImageOps.exif_transpose(im).convert("RGB"))
    ax.set_title(f"{row['class']}\n{row['path']}", fontsize=8)
    ax.axis("off")
plt.tight_layout()


## Prompt notes: final prompts are recorded below

The adapter card's reference instruction is `Enhance the aesthetic quality of this image.` The initial prompt appends object-preservation constraints. Replace each class value below with the final prompt the user writes. Keep instructions limited to enhancement; requests that change class identity, count, object shape, labels, or scene would undermine the dataset label.

Prompt ideas for the user to consider (examples only, not executed):
- bottles: improve lighting and composition while preserving the bottle's exact label, silhouette, cap, and count
- cups: improve lighting and clarity while preserving handle shape, color, print, and cup count
- pens: improve lighting and composition while preserving pen geometry, tip, cap, markings, and count
- scissors: improve lighting and clarity while preserving blade shape, handle color, and count

The notebook requires all four prompts and an explicit flag before model loading or generation.


In [ ]:
# Final prompts supplied by the user. Generation remains disabled pending runtime access.
PROMPTS_FINALIZED = True
RUN_GENERATION = False  # Set True after GPU and authenticated weight access preflight succeed.
HF_WEIGHT_ACCESS_VERIFIED = False
PROMPTS = {
    "bottles": "Improve the lighting and visual clarity of this bottle photograph. Reduce distracting glare while preserving the bottle’s shape, colour, cap and label. Keep the background and camera angle unchanged. Do not add or remove objects.",
    "cups": "Improve the exposure and clarity of this cup. Soften harsh shadows while preserving the cup’s shape, colour, handle and any printed design. Keep the background and camera angle unchanged. Do not add or remove objects.",
    "pens": "Improve the lighting and contrast of this pen photograph so its outline is easier to see. Preserve its shape, colour, tip, cap and visible markings. Keep the background and camera angle unchanged. Do not add or remove objects.",
    "scissors": "Improve the lighting and clarity of this scissors photograph. Reduce glare on the blades while preserving both blades, the handles, their colours and the opening angle. Keep the background unchanged. Do not add or remove objects.",
}

if not PROMPTS_FINALIZED:
    print("Prompts are finalized; generation remains separately disabled until GPU and gated model access are ready.")
else:
    assert set(PROMPTS) == set(CLASS_ORDER) and all(isinstance(PROMPTS[k], str) and PROMPTS[k].strip() for k in CLASS_ORDER)
    print("Prompt check passed.")


In [ ]:
# GPU capability and small kernel smoke checks; no generator weights are loaded.
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime first.")
props = torch.cuda.get_device_properties(0)
print({"gpu": props.name, "vram_gib": round(props.total_memory/1024**3,2), "compute_capability": torch.cuda.get_device_capability(), "native_bf16": torch.cuda.is_bf16_supported(including_emulation=False), "torch": torch.__version__})
for dtype in [torch.bfloat16, torch.float16]:
    try:
        x = torch.randn((256,256), device="cuda", dtype=dtype)
        y = x @ x.T
        z = x.to(torch.float8_e4m3fn).to(dtype)
        torch.cuda.synchronize()
        print(str(dtype), "matmul/FP8-cast finite:", bool(torch.isfinite(y).all() and torch.isfinite(z).all()))
        del x, y, z
    except Exception as error:
        print(str(dtype), type(error).__name__, str(error)[:300])
torch.cuda.empty_cache()


In [ ]:
# Generation status. Prompts are final; model generation remains explicitly off by default.
if not PROMPTS_FINALIZED:
    print("Blocked: final prompts are not confirmed.")
elif not RUN_GENERATION:
    print("Ready for GPU and authenticated weight-access preflight; generation stays off until those checks pass.")
else:
    print("Generation explicitly enabled by the user.")


In [ ]:
# Install the pinned DiffSynth runtime only after the GPU and authenticated weight-access preflights pass and generation is enabled.
# Colab's preinstalled CUDA PyTorch is retained.
if RUN_GENERATION:
    subprocess.run([sys.executable, "-m", "pip", "install", "diffsynth==2.1.12", "huggingface_hub", "modelscope"], check=True)
else:
    print("Skipped DiffSynth installation; RUN_GENERATION=False.")


In [ ]:
# Read-only Hugging Face authentication. Uses only the Colab secret; never starts OAuth/login.
HF_WEIGHT_ACCESS_VERIFIED = False

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if not hf_token or not isinstance(hf_token, str):
    raise RuntimeError(
        "HF_TOKEN is missing. In Hugging Face Settings > Access Tokens, create a token "
        "with role `read`, or a fine-grained token scoped to read-only access for the "
        "three required model repositories. Accept the FLUX.1-Kontext-dev and FLUX.1-dev "
        "gated model terms. In Colab, open Secrets (key icon), add HF_TOKEN, paste the "
        "token there, enable notebook access, then rerun this cell. Do not paste it into code."
    )

try:
    import huggingface_hub
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "huggingface_hub"], check=True)

from huggingface_hub import HfApi, hf_hub_url
from huggingface_hub.file_download import get_hf_file_metadata

# whoami returns limited token metadata; only inspect the role, never print the response.
try:
    auth_info = HfApi().whoami(token=hf_token)
    token_role = ((auth_info.get("auth") or {}).get("accessToken") or {}).get("role")
except Exception as exc:
    del hf_token
    raise RuntimeError(
        f"Hugging Face token validation failed ({type(exc).__name__}). Check that HF_TOKEN "
        "is a valid user access token stored as a Colab secret. The token value and server "
        "response were suppressed."
    ) from None

if token_role not in {"read", "fineGrained"}:
    del hf_token
    raise RuntimeError(
        f"This workflow requires a read-only Hugging Face token; detected role: {token_role!r}. "
        "Create a `read` token, or a fine-grained token with read-only access only to the "
        "required model repositories, then update the HF_TOKEN Colab secret."
    )
if token_role == "fineGrained":
    print("Token role is fineGrained. Confirm its settings grant read-only access to the required model repositories.")
else:
    print("Token role is read.")

# Runtime-only environment value lets later Hugging Face/DiffSynth downloads reuse the secret.
# It is not written to notebook source, files, Drive, or Git credentials.
os.environ["HF_TOKEN"] = hf_token

BASE_ID = "black-forest-labs/FLUX.1-Kontext-dev"
AUX_ID = "black-forest-labs/FLUX.1-dev"
ADAPTER_ID = "DiffSynth-Studio/FLUX.1-Kontext-dev-lora-ArtAug"
required_weight_files = [
    (BASE_ID, "flux1-kontext-dev.safetensors"),
    (AUX_ID, "text_encoder/model.safetensors"),
    (AUX_ID, "text_encoder_2/model-00001-of-00002.safetensors"),
    (AUX_ID, "text_encoder_2/model-00002-of-00002.safetensors"),
    (AUX_ID, "ae.safetensors"),
    (ADAPTER_ID, "model.safetensors"),
]
weight_access = []
for repo_id, filename in required_weight_files:
    try:
        metadata = get_hf_file_metadata(hf_hub_url(repo_id, filename), token=hf_token)
    except Exception as exc:
        del hf_token
        raise RuntimeError(
            f"Read access check failed for {repo_id}/{filename} ({type(exc).__name__}). "
            "Confirm you accepted the gated FLUX model terms and that the read-only token "
            "is scoped to these repositories. Error details were suppressed to prevent "
            "credential disclosure."
        ) from None
    weight_access.append({
        "repo_id": repo_id,
        "filename": filename,
        "commit_hash": metadata.commit_hash,
        "size_bytes": metadata.size,
    })
    print("verified read access", repo_id, filename, metadata.commit_hash, metadata.size)

del hf_token
HF_WEIGHT_ACCESS_VERIFIED = True
print("All required weight-file read checks passed; no model weights were downloaded in this cell.")

In [ ]:
# Load official DiffSynth FLUX.1-Kontext pipeline and released ArtAug adapter.
# The artifact card shows `flux_image_new`; current official Kontext examples use `flux_image`.
if not PROMPTS_FINALIZED or not RUN_GENERATION or not HF_WEIGHT_ACCESS_VERIFIED:
    raise RuntimeError("Model loading is blocked until authenticated weight access passed and RUN_GENERATION=True after GPU checks.")
os.environ["DIFFSYNTH_DOWNLOAD_SOURCE"] = "huggingface"
os.environ["DIFFSYNTH_MODEL_BASE_PATH"] = "/content/model-cache"
import diffsynth
from diffsynth.pipelines.flux_image import FluxImagePipeline, ModelConfig
from huggingface_hub import snapshot_download, HfApi

# Follow the official DiffSynth low-VRAM Kontext example's layerwise CPU offload recipe.
vram_config = {
    "offload_dtype": torch.float8_e4m3fn, "offload_device": "cpu",
    "onload_dtype": torch.float8_e4m3fn, "onload_device": "cpu",
    "preparing_dtype": torch.float8_e4m3fn, "preparing_device": "cuda",
    "computation_dtype": torch.bfloat16, "computation_device": "cuda",
}
model_configs = [
    ModelConfig(model_id=BASE_ID, origin_file_pattern="flux1-kontext-dev.safetensors", download_source="huggingface", **vram_config),
    ModelConfig(model_id=AUX_ID, origin_file_pattern="text_encoder/model.safetensors", download_source="huggingface", **vram_config),
    ModelConfig(model_id=AUX_ID, origin_file_pattern="text_encoder_2/*.safetensors", download_source="huggingface", **vram_config),
    ModelConfig(model_id=AUX_ID, origin_file_pattern="ae.safetensors", download_source="huggingface", **vram_config),
]
pipe = FluxImagePipeline.from_pretrained(
    torch_dtype=torch.bfloat16, device="cuda", model_configs=model_configs,
    vram_limit=torch.cuda.mem_get_info(0)[1] / 1024**3 - 0.5,
)
# Pin the adapter snapshot to the exact commit whose file metadata was verified above.
adapter_revision = next(item["commit_hash"] for item in weight_access if item["repo_id"] == ADAPTER_ID)
adapter_dir = snapshot_download(ADAPTER_ID, revision=adapter_revision, allow_patterns=["model.safetensors"], token=True)
adapter_path = Path(adapter_dir) / "model.safetensors"
pipe.load_lora(pipe.dit, str(adapter_path), alpha=1)
adapter_sha256 = sha256_file(adapter_path)
print({"diffsynth_version": getattr(diffsynth, "__version__", "unknown"),
       "adapter_revision": adapter_revision, "adapter_sha256": adapter_sha256})


In [ ]:
# Generate the first image as a smoke pilot; save each image and manifest entry immediately.
# Increase MAX_IMAGES_PER_RUN only after reviewing the first image. Existing completed IDs are skipped.
if not PROMPTS_FINALIZED or not RUN_GENERATION or not HF_WEIGHT_ACCESS_VERIFIED:
    raise RuntimeError("Generation is locked; pass the authenticated HF access check, confirm GPU, then set RUN_GENERATION=True.")
from datetime import datetime, timezone
from PIL import Image, ImageOps
from IPython.display import display

OUTPUT_DIR = Path("/content/artaug_pilot")
images_dir = OUTPUT_DIR / "images"
images_dir.mkdir(parents=True, exist_ok=True)
MAX_IMAGES_PER_RUN = 1  # initial one-image pilot; review it, then increase to generate the remaining examples
manifest_out = OUTPUT_DIR / "manifest.json"
if manifest_out.exists():
    bundle = json.loads(manifest_out.read_text(encoding="utf-8"))
    examples = bundle["examples"]
else:
    examples = []
completed = {row["id"] for row in examples}
remaining_budget = MAX_IMAGES_PER_RUN

generator = {
    "name": "DiffSynth-Studio FLUX.1-Kontext ArtAug image-editing LoRA",
    "adapter_id": ADAPTER_ID, "adapter_revision": adapter_revision, "adapter_sha256": adapter_sha256,
    "base_model_id": BASE_ID, "base_model_revision_observed": next(item["commit_hash"] for item in weight_access if item["repo_id"] == BASE_ID),
    "aux_model_id": AUX_ID, "aux_model_revisions_observed": sorted({item["commit_hash"] for item in weight_access if item["repo_id"] == AUX_ID}),
    "runtime": "diffsynth==2.1.12", "torch": torch.__version__,
    "diffsynth_api": "FluxImagePipeline from diffsynth.pipelines.flux_image",
    "license_note": "The base FLUX.1-Kontext-dev / FLUX.1-dev terms apply; non-commercial license.",
    "synthetic_images": True, "seed_policy": f"{PILOT_SEED} + selection index",
    "pilot_source_qc_exclusions": PILOT_SOURCE_QC_EXCLUSIONS,
    "resolution_policy": "preserve source aspect; longest output side at most 768 px; dimensions rounded to multiples of 16",
    "steps": 30, "embedded_guidance": 2.5,
    "note": "Base/aux files are fetched through DiffSynth ModelConfig by repo ID; observed commits are recorded, but DiffSynth ModelConfig does not pin their revisions in this notebook.",
}
if manifest_out.exists():
    if bundle.get("source_manifest_sha256") != SOURCE_MANIFEST_SHA256:
        raise RuntimeError("Existing pilot bundle was made from a different frozen manifest")
    if bundle.get("generator") != generator:
        raise RuntimeError("Existing pilot bundle generator/runtime differs; start a new output folder")
for index, row in enumerate(SELECTED):
    identifier = hashlib.sha256(f"{row['class']}\n{row['path']}".encode()).hexdigest()[:16]
    if identifier in completed:
        continue
    if remaining_budget <= 0:
        break
    seed = PILOT_SEED + index
    original_rel, enhanced_rel = f"images/{identifier}_original.jpg", f"images/{identifier}_enhanced.jpg"
    original_out, enhanced_out = OUTPUT_DIR / original_rel, OUTPUT_DIR / enhanced_rel
    source_path = SOURCE_DIR / row["path"]
    shutil.copyfile(source_path, original_out)
    with Image.open(source_path) as im:
        source_image = ImageOps.exif_transpose(im).convert("RGB")
    prompt = PROMPTS[row["class"]].strip()
    scale = min(768 / max(source_image.size), 1.0)
    out_width = max(16, round(source_image.width * scale / 16) * 16)
    out_height = max(16, round(source_image.height * scale / 16) * 16)
    generation_started_utc = datetime.now(timezone.utc).isoformat()
    started = time.perf_counter()
    edited = pipe(prompt=prompt, kontext_images=source_image, height=out_height, width=out_width, num_inference_steps=30, embedded_guidance=2.5, seed=seed)
    elapsed = time.perf_counter() - started
    edited.save(enhanced_out, format="JPEG", quality=95, optimize=True)
    example = {
        "id": identifier, "class_name": row["class"],
        "source_manifest_path": row["path"], "source_sha256": row["sha256"],
        "original": original_rel, "original_sha256": sha256_file(original_out),
        "enhanced": enhanced_rel, "enhanced_sha256": sha256_file(enhanced_out),
        "prompt": prompt, "seed": seed, "approved": False,
        "generation": {"started_utc": generation_started_utc, "duration_seconds": round(elapsed, 3),
                        "gpu": props.name, "requested_resolution": [out_width, out_height], "actual_resolution": list(edited.size), "steps": 30, "embedded_guidance": 2.5},
    }
    examples.append(example)
    completed.add(identifier)
    bundle = {"schema_version": 1, "generator": generator,
              "source_manifest_sha256": SOURCE_MANIFEST_SHA256, "examples": examples}
    manifest_out.write_text(json.dumps(bundle, indent=2, ensure_ascii=False)+"\n", encoding="utf-8")
    print(f"{len(completed)}/20 {row['class']} {row['path']} seed={seed} seconds={elapsed:.1f}")
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(source_image); ax[0].set_title("Original")
    ax[1].imshow(edited); ax[1].set_title("ArtAug edit — unapproved")
    for a in ax: a.axis("off")
    plt.show()
    remaining_budget -= 1
print("Current pilot bundle:", OUTPUT_DIR, "entries:", len(examples), "(all start unapproved)")


## Manual review and app import

Review every original/edit pair at full size. Only after manual review, set `approved: true` for acceptable edits in `manifest.json`; do not approve any image that changes class identity, object count, silhouette, defining labels, or scene content. Keep rejected images in the pilot output for audit, with `approved: false`.

The first run generates one image and saves the manifest incrementally. Review it before increasing `MAX_IMAGES_PER_RUN` to continue. The completed images are displayed inline and saved under `/content/artaug_pilot`; download that folder as a ZIP from the Colab file browser before the runtime expires. In a local checkout of this project, run `python tools/import_artaug.py --bundle <folder>`. The importer checks the frozen source manifest, source membership in the train split, file hashes, safe paths, and approval flags. It copies only approved image pairs to app assets. This does not retrain the classifier or establish a metric improvement.
